# Cluster results — locations and focal mechanisms

A single, cluster-parameterized notebook to view a cluster's **located catalog** and its
**focal mechanisms together**. Set the params below and run top to bottom.

Focal mechanisms require a **phasenet_plus** run (the PhaseNet+ picker emits first-motion polarity
+ S/P amplitude that SKHASH needs); point `RUN_SUFFIX` at that run's output tree. Locations alone
work for any picker.

In [ ]:
%matplotlib inline
import os, sys
# Assumes the notebook runs from pipeline/notebooks/ ; otherwise set PYTHONPATH=<repo root>.
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..")))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from pipeline import config, viz

# ------------------------------- PARAMS (edit & re-run) -------------------------------
CLUSTER    = "jangsung"        # gwangyang | kimcheon | jangsung | gyeongju
RUN_SUFFIX = "_pnplus"          # output tree = runs/<cluster><suffix>; "" = the default run
VELMODEL   = "kim1983"          # velocity model whose .sum / mechanisms to show

cfg0 = config.load_cluster(CLUSTER)
cfg  = config.tune(cfg0, output_root=os.path.join(config.RUNS_ROOT, f"{CLUSTER}{RUN_SUFFIX}")) \
       if RUN_SUFFIX else cfg0
print(f"{cfg.region}: outputs -> {cfg.output_root}")

## 1. Locations

Three location stages, each with fewer events than the last: **`.sum`** is every event HYPOINVERSE
locates absolutely; **dt.ct** is the HypoDD catalog relocation, which keeps only events with enough
catalog differential-time links (isolated events drop); **dt.cc** is the HypoDD cross-correlation
relocation, which further keeps only events whose waveforms correlate well. **dt.cc is the high-end
product** (errors of metres vs tens of metres for dt.ct) and is the headline relocation below. Counts
shrink `.sum ≥ dt.ct ≥ dt.cc` but not strictly — each HypoDD run re-clusters independently.

In [ ]:
display(viz.relocation_counts(cfg, VELMODEL))

In [ ]:
viz.map_catalog(cfg, velmodel=VELMODEL, source="sum"); plt.show()
viz.depth_sections(cfg, velmodel=VELMODEL, source="sum"); plt.show()
viz.cumulative_events(cfg, velmodel=VELMODEL); plt.show()

In [ ]:
# headline relocated catalog (dt.cc if present, else dt.ct)
if os.path.exists(os.path.join(config.dtct_dir(cfg), "hypoDD.reloc")):
    viz.map_catalog(cfg, velmodel=VELMODEL, source="reloc"); plt.show()
    if os.path.exists(os.path.join(config.dtcc_dir(cfg), "hypoDD.reloc")):
        viz.compare_epicenters(cfg, velmodel=VELMODEL); plt.show()   # dt.ct vs dt.cc
else:
    print("No HypoDD reloc for this run — showing absolute (.sum) locations only.")

## 2. Picks and first-motion polarity

PhaseNet+ picks carry a first-motion **polarity** (up/down) — the input to the focal-mechanism inversion.
`plot_3c` shows the three components with P (red) / S (blue) and the P polarity marked on the vertical;
`plot_polarities` is a P-aligned first-motion record section sorted by azimuth (red = up, blue = down),
i.e. the up/down-vs-azimuth pattern that constrains the mechanism.

In [ ]:
# a representative event: the best-quality mechanism, else the first picks CSV
import glob as _glob
_tbl = viz.mechanism_table(cfg, VELMODEL)
if len(_tbl):
    SAMPLE_EVENT = str(_tbl.sort_values("quality").iloc[0].event_id)
else:
    _pf = sorted(_glob.glob(os.path.join(config.picks_dir(cfg), "*_picks.csv")))
    SAMPLE_EVENT = os.path.basename(_pf[0]).split("_")[0] if _pf else None
print("Sample event:", SAMPLE_EVENT)
viz.plot_3c(cfg, SAMPLE_EVENT); plt.show()
viz.plot_polarities(cfg, SAMPLE_EVENT); plt.show()

## 3. Focal mechanisms

The table is one row per event (best quality kept). `map_mechanisms` shows the **locations and
focal mechanisms together**: located epicenters as depth-coloured dots, with the high-confidence
(quality A/B) beachballs offset on a ring around the cluster (leader line to each true epicenter)
so a tight cluster stays legible.

In [ ]:
tbl = viz.mechanism_table(cfg, VELMODEL)
display(tbl)
viz.map_mechanisms(cfg, VELMODEL); plt.show()

In [ ]:
# per-event beachball gallery (SKHASH plots) for the high-confidence events
from matplotlib import image as mpimg
mech = pd.read_csv(config.fm_mech_csv(cfg, VELMODEL)).drop_duplicates("event_id")
e2c  = dict(zip(mech.event_id.astype(str), mech.cuspid.astype(int)))
# prefer the high-confidence (A/B) events; if none (e.g. coverage-limited clusters), show the
# best-graded events anyway so the (low-confidence) mechanisms are still visible
hi   = tbl[tbl.quality.isin(cfg.fm_quality_keep)] if len(tbl) else tbl
sel  = (hi if len(hi) else tbl.sort_values("quality")).head(9)
ids  = list(sel.event_id.astype(str))
out  = config.fm_out_dir(cfg, VELMODEL)
if ids:
    ncol = min(3, len(ids)); nrow = (len(ids) + ncol - 1) // ncol
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.6 * ncol, 3.8 * nrow), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, eid in zip(axes.ravel(), ids):
        r = sel[sel.event_id.astype(str) == eid].iloc[0]
        png = os.path.join(out, f"{e2c.get(eid)}.png")
        if os.path.exists(png):
            ax.imshow(mpimg.imread(png))
        ax.set_title(f"{eid}   {r.quality}\ns/d/r = {r.strike:.0f}/{r.dip:.0f}/{r.rake:.0f}",
                     fontsize=9)
    fig.suptitle(f"{cfg.region} — high-confidence focal mechanisms (SKHASH beachballs)", fontsize=11)
    plt.tight_layout(); plt.show()
else:
    print("No high-confidence (A/B) mechanisms for this run "
          "(needs a phasenet_plus focal_mechanism run).")

## 4. Seismicity in fault coordinates

`fault_sections` rotates the dt.cc relocated catalog into the fault frame and shows four panels:
a **fault-plane map view** (with the strike line, the perpendicular, and the focal-mechanism beachball),
an **along-strike** depth section (A–A'), an **across-strike** depth section (B–B', dashed line = dip),
and a **fault-plane (along-dip) view**. Markers are coloured by origin time (so migration is visible)
and sized by magnitude. The orientation is the **best-fit plane of the relocated cloud** (data-driven,
via SVD), with the focal mechanism overlaid only for comparison — pass `strike=`/`dip=` to override.
A tight across-strike spread indicates a near-planar fault.

In [ ]:
viz.fault_sections(cfg, VELMODEL); plt.show()

## 5. Interpretation

- **Quality A/B** = well-constrained ("fairly high confidence"); C/D are under-constrained and shown
  for context only. SKHASH grades on polarity misfit, station-distribution ratio, azimuthal/takeoff
  gaps, and mechanism probability.
- **Polarity** (vertical first motion) is the robust signal; the vertical-component **S/P ratio** is a
  secondary enhancement (`cfg.fm_use_sp_ratio`). Re-run with `fm_use_sp_ratio=False` for a
  polarity-only comparison.
- Consistent beachballs across events indicate a coherent source process on a common fault geometry.
- **Fault frame vs mechanism.** The section orientation is the relocation cloud's own best-fit plane,
  and the focal mechanism is overlaid for comparison. When the two agree (e.g. Gwangyang, where the
  best-fit strike ≈ the mechanism's nodal plane), the fault geometry is well determined. When they
  disagree, read the section's strike/dip header: an under-constrained cluster (e.g. **Jangsung** — only
  a few dt.cc events and a grade-D mechanism) still gets a data-driven fault frame, but the mechanism is
  not reliable there, so the section is indicative only.
- To regenerate: run `picking` (`--picker phasenet_plus`), `hypoinverse`, then the `focal_mechanism`
  stage for this cluster (see the top-level README).